# Phase 1 : 
## goal-conditioned observations on PC-Gym’s native distillation environment,SAC baseline, fixed/random/hand-crafted-curriculum baselines, an adaptive non-LLM curriculum baseline , NMPC benchmark.

#### Experiment A:  Curriculum ordering (answers RQ1 and RQ2 cleanly). 
A single fixed
goal set G is defined in advance, drawing on all three goal types from Section 5.1 (e.g. bottoms-
only targets at 90/92/95/98%, distillate-only targets at the same levels, plus a small number of
combined targets). Every condition in Section 9.1 trains and is evaluated on exactly this same set
G; only the order in which goals are presented during training differs between conditions. This
isolates ordering as the only independent variable, so a sample-efficiency comparison between
conditions is a fair, like-for-like comparison , the thing being tested is purely whether a smarter
curriculum order helps the agent reach the same final goals faster

#### 1) goal-conditioned observations on PC-Gym’s native distillation environment

In [5]:
import time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from pcgym import make_env
from pcgym.model_classes import distillation_column

In [7]:
# Fixing bug in casadi (unpacking)

def corrected_call(self,x,u):
    X0, X1, X2, X3, Xf, X4, X5, X6, Xb = (x[i] for i in range(9))
    R, Fe = u[0], u[1]

    L = R * self.D
    V = (R + 1) * self.D
    L_dash = L + self.q * Fe
    V_dash = V + (1 - self.q) * Fe
    W = Fe - self.D

    def Y(X):
        return (self.alpha * X) / (1 + (self.alpha - 1) * X)

    Y1, Y2, Y3, Yf = Y(X1), Y(X2), Y(X3), Y(Xf)
    Y4, Y5, Y6, Yb = Y(X4), Y(X5), Y(X6), Y(Xb)

    dxdt = [
        (1 / self.M0) * ((V * Y1) - (L + self.D) * X0),
        (1 / self.M) * (L * (X0 - X1) + V * (Y2 - Y1)),
        (1 / self.M) * (L * (X1 - X2) + V * (Y3 - Y2)),
        (1 / self.M) * (L * (X2 - X3) + V * (Yf - Y3)),
        (1 / self.M) * (L * X3 - L_dash * Xf + V_dash * Y4 - V * Yf + Fe * self.X_feed),
        (1 / self.M) * (L_dash * (Xf - X4) + V_dash * (Y5 - Y4)),
        (1 / self.M) * (L_dash * (X4 - X5) + V_dash * (Y6 - Y5)),
        (1 / self.M) * (L_dash * (X5 - X6) + V_dash * (Yb - Y6)),
        (1 / self.Mb) * (L_dash * X6 - W * Xb - V_dash * Yb),
    ]
    return dxdt

distillation_column.__call__=corrected_call
print("pcgym distillation_column corrected.")




pcgym distillation_column corrected.


In [16]:
# FIXED GOAL SET ( in experiment A: only purity goal)

PURITY_LEVELS = [0.90, 0.92, 0.95, 0.98]
COMBINED_PAIRS = [(0.95, 0.95), (0.97, 0.96), (0.95, 0.98)]  # (distillate, bottoms)
GOAL_TYPES = ["bottoms", "distillate", "combined"]

def make_goal(gid, gtype, purity_distillate=None, purity_bottoms=None):
    return {"id":gid,"type":gtype,"purity_distillate":purity_distillate,"purity_bottoms": purity_bottoms,
            "X0_target":purity_distillate if purity_distillate is not None else None,
            "Xb_target":purity_bottoms if purity_bottoms is not None else None}

def build_goal_set():
    G=[]
    for p in PURITY_LEVELS:
        G.append(make_goal(f"bottoms_{int(p*100)}%","bottoms",purity_bottoms=p))
        G.append(make_goal(f"distillate_{int(p*100)}%","distillate",purity_distillate=p))
    for c in COMBINED_PAIRS:
        G.append(make_goal(f"combined_{int(c[0]*100)}%_{int(c[1]*100)}%","combined",purity_distillate=c[0],purity_bottoms=c[1]))

    return G

def goal_type_onehot(goal):
    v=np.zeros(len(GOAL_TYPE))
    v[GOAL_TYPE.index(goal.type)]=1.0
    return v


G = build_goal_set()
print(f"|G| = {len(G)}")
for g in G:
    print(f"  {g['id']:16s} type={g['type']:10s} X0_target={g['X0_target']}  Xb_target={g['Xb_target']}")

    
    
     
            




|G| = 11
  bottoms_90%      type=bottoms    X0_target=None  Xb_target=0.9
  distillate_90%   type=distillate X0_target=0.9  Xb_target=None
  bottoms_92%      type=bottoms    X0_target=None  Xb_target=0.92
  distillate_92%   type=distillate X0_target=0.92  Xb_target=None
  bottoms_95%      type=bottoms    X0_target=None  Xb_target=0.95
  distillate_95%   type=distillate X0_target=0.95  Xb_target=None
  bottoms_98%      type=bottoms    X0_target=None  Xb_target=0.98
  distillate_98%   type=distillate X0_target=0.98  Xb_target=None
  combined_95%_95% type=combined   X0_target=0.95  Xb_target=0.95
  combined_97%_96% type=combined   X0_target=0.97  Xb_target=0.96
  combined_95%_98% type=combined   X0_target=0.95  Xb_target=0.98


### Goal-conditioned observation and reward wrapper

**Observation** $o_g = [x,\ x_{sp},\ d,\ g]$:
- $x$: the 9 process states (Section 4.2).
- $x_{sp}$: the active goal's numeric targets, carried natively by pcgym's
  `SP` mechanism (constant-over-episode setpoint per tracked variable, `X0`
  and `Xb`). An untracked target uses a **&minus;1 placeholder** (outside the
  physical $[0,1]$ range), so "no target on this variable" is never confused
  with "target = 0".
- $d$: disturbances. Phase 1 trains on the **nominal** process only — no
  disturbances yet (those arrive with the LLM scenario generator, Section 6 /
  Phase 3). The slot is reserved so the wrapper's shape doesn't need to
  change later.
- $g$: one-hot goal **type** (bottoms / distillate / combined). The numeric
  targets are already carried by $x_{sp}$; $g$ supplies the type information
  Section 8 asks for.

**Reward** (Section 5.3), computed once at the terminal step:
$$r_T = r(\cdot) + \lambda_{bonus}\mathbb{1}[\cdot] - \lambda_{energy}R^2 -
\lambda_{flow}\max(0,F_{min}-F) - \lambda_{constraint}\cdot\text{violation}$$

Two adaptations, both because pcgym's `distillation_column` does not model
pressure:
- the **operating constraint** (Section 5.2) is a vapour-flow ceiling
  $V=(R+1)D$ — a standard flooding-style capacity constraint, used as the
  concrete stand-in for the "pressure between 0.95–1.05 bar" example in the
  plan text;
- the **flow-shortfall penalty** applies to $F$ (the actually-controlled feed
  rate), since $D$ is a fixed model parameter here, not a decision variable.


In [ ]:
N_STEPS = 40
TSIM = 160.0
X0_STATE = np.array([0.93, 0.85, 0.75, 0.60, 0.20, 0.15, 0.10, 0.06, 0.03])
A_LOW  = np.array([1.0, 50.0])   # [R, F]
A_HIGH = np.array([15.0, 150.0])

SIGMA, EPSILON = 0.10, 0.02
LAMBDA_BONUS, LAMBDA_ENERGY, LAMBDA_FLOW, LAMBDA_CONSTRAINT = 1.0, 0.002, 0.02, 1.0
F_MIN, V_MAX = 60.0, 1400.0

def build_model():
    return distillation_column(D=100.0, q=1.0, alpha=5.0, X_feed=0.2, M0=2000.0, Mb=2000.0, M=2000.0)

def vapour_flow_constraint(state,u):
    R = u[0]
    return bool((R + 1) * 100.0 > V_MAX)